In [6]:
import numpy as np
scaled_factors = np.load("../result/scaled_factors.npy")
ability_df_scaled = np.load("../result/ability_df_scaled.npy")

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# --- 1. Simulate Data (Same as before for consistency) ---
# In your real-world case, you would load your data here.
num_subjects = ability_df_scaled.shape[0]
num_items = scaled_factors.shape[0]
num_dims = scaled_factors.shape[1]  # <-- Set this to the number of skills from your SVD

true_thetas = ability_df_scaled.values
true_discriminations = np.random.lognormal(0, 0.5, size=(num_items, num_dims)) # Use lognormal for positive discriminations
true_difficulties = np.random.randn(num_items)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

responses_list = []
for s_idx in range(num_subjects):
    for i_idx in range(num_items):
        logit = np.dot(true_thetas[s_idx], true_discriminations[i_idx]) - true_difficulties[i_idx]
        prob = sigmoid(logit)
        response = 1 if np.random.random() < prob else 0
        responses_list.append({'subject_id': s_idx, 'item_id': i_idx, 'response': response})

response_df = pd.DataFrame(responses_list)

# --- 2. Prepare Data for PyTorch ---
# We need tensors for subject indices, item indices, and the actual responses.
subject_indices = torch.tensor(response_df['subject_id'].values, dtype=torch.long)
item_indices = torch.tensor(response_df['item_id'].values, dtype=torch.long)
responses = torch.tensor(response_df['response'].values, dtype=torch.float32)

# --- 3. Define the MIRT Model in PyTorch ---
class MIRTModel(nn.Module):
    def __init__(self, num_subjects, num_items, num_dims):
        super().__init__()
        # Person parameters (abilities)
        self.thetas = nn.Parameter(torch.randn(num_subjects, num_dims))
        # Item parameters (discrimination and difficulty)
        self.discriminations = nn.Parameter(torch.randn(num_items, num_dims))
        self.difficulties = nn.Parameter(torch.randn(num_items))

    def forward(self, subject_indices, item_indices):
        # Look up the parameters for the subjects and items in the current batch
        subject_thetas = self.thetas[subject_indices]
        item_discriminations = self.discriminations[item_indices]
        item_difficulties = self.difficulties[item_indices]

        # Compute the dot product of person abilities and item discriminations
        dot_product = torch.sum(subject_thetas * item_discriminations, dim=1)

        # Apply the MIRT formula to get the log-odds
        logits = dot_product - item_difficulties

        # Apply the sigmoid function to get probabilities
        return torch.sigmoid(logits)

# --- 4. Training the Model ---
# Hyperparameters
learning_rate = 0.01
epochs = 1000 # More epochs for better convergence
batch_size = 8192 # Use a large batch size for stability

# Instantiate the model
model = MIRTModel(num_subjects, num_items, num_dims)

# Loss and optimizer
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Starting model training... 🚀")
for epoch in range(epochs):
    # Simple batching for demonstration
    permutation = torch.randperm(len(subject_indices))
    for i in range(0, len(subject_indices), batch_size):
        indices = permutation[i:i+batch_size]
        batch_subjects, batch_items, batch_responses = subject_indices[indices], item_indices[indices], responses[indices]
        
        # 1. Forward pass: compute predicted probabilities
        pred_probs = model(batch_subjects, batch_items)

        # 2. Compute loss
        loss = loss_fn(pred_probs, batch_responses)

        # 3. Zero gradients, backward pass, and update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("Training complete!")

# --- 5. Inspect the Results ---
# Convert the learned parameters back to NumPy arrays for analysis
estimated_thetas = model.thetas.detach().numpy()
estimated_discriminations = model.discriminations.detach().numpy()
estimated_difficulties = model.difficulties.detach().numpy()

print("\n--- Model Parameters Learned ---")
print(f"Shape of estimated person abilities (thetas): {estimated_thetas.shape}")
print(f"Shape of estimated item discriminations: {estimated_discriminations.shape}")
print(f"Shape of estimated item difficulties: {estimated_difficulties.shape}")

print("\nExample ability vector for User 0:", np.round(estimated_thetas[0], 2))
print("Example discrimination vector for Item 0:", np.round(estimated_discriminations[0], 2))
print("Example difficulty for Item 0:", np.round(estimated_difficulties[0], 2))

Starting model training... 🚀
Epoch [100/1000], Loss: 0.4267
Epoch [200/1000], Loss: 0.4259
Epoch [300/1000], Loss: 0.4132
Epoch [400/1000], Loss: 0.4155
Epoch [500/1000], Loss: 0.4200
Epoch [600/1000], Loss: 0.4300
Epoch [700/1000], Loss: 0.4173
Epoch [800/1000], Loss: 0.4271
Epoch [900/1000], Loss: 0.4250
Epoch [1000/1000], Loss: 0.4097
Training complete!

--- Model Parameters Learned ---
Shape of estimated person abilities (thetas): (1000, 3)
Shape of estimated item discriminations: (100, 3)
Shape of estimated item difficulties: (100,)

Example ability vector for User 0: [0.3  0.16 0.57]
Example discrimination vector for Item 0: [-0.02  0.47  0.58]
Example difficulty for Item 0: 0.21


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import pearsonr
from torchmetrics import AUROC
from tqdm import tqdm

# ===================================================================
# == Step 1: Load and Prepare All Data
# ===================================================================
print("--- Step 1: Loading and preparing data ---")
resmat = pd.read_pickle("data/resmat.pkl")
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load Embeddings ---
with open("data/embed_meta-llama_Llama-3.1-8B-Instruct.pkl", "rb") as f:
    df_embed = pickle.load(f)

question_to_emb = dict(zip(df_embed["question"], df_embed["embedding"]))
questions = resmat.columns.get_level_values("input.text").tolist()
embed_dim = df_embed["embedding"].iloc[0].shape[0]
embeddings_np = np.array([question_to_emb.get(q, np.zeros(embed_dim)) for q in questions])

# --- Create Train/Test Split Masks ---
data_withnan = torch.tensor(resmat.values, dtype=torch.float32)
data_idtor = torch.isfinite(data_withnan)
valid_condition = False
while not valid_condition:
    train_idtor = torch.bernoulli(data_idtor.float() * 0.8).int()
    valid_condition = (train_idtor.sum(dim=1) > 0).all() and (train_idtor.sum(dim=0) > 0).all()
test_idtor = data_idtor.int() - train_idtor

# --- Create DataLoader for Training ---
# Get indices and values for known training data points
train_indices = torch.where(train_idtor.bool())
train_responses = data_withnan[train_indices]
# The dataset will provide (model_idx, item_idx, response)
train_dataset = TensorDataset(train_indices[0], train_indices[1], train_responses)
train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True)

# Convert data to tensors
data_tensor = torch.nan_to_num(data_withnan, nan=0.0).to(device)
embeddings_tensor = torch.tensor(embeddings_np, dtype=torch.float32).to(device)

# ===================================================================
# == Step 2: Define the Neural IRT (1PL) Model
# ===================================================================
class NeuralIRT1PL(nn.Module):
    def __init__(self, n_models, embedding_dim):
        super(NeuralIRT1PL, self).__init__()
        # Learnable ability parameter for each of the 183 models
        self.thetas = nn.Parameter(torch.randn(n_models, 1))
        
        # A neural network to predict difficulty from an embedding
        self.difficulty_net = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1) # Outputs a single difficulty value
        )

    def forward(self, model_indices, item_embeddings):
        # Get the ability for each model in the batch
        model_ability = self.thetas[model_indices]
        # Predict the difficulty for each item in the batch
        item_difficulty = self.difficulty_net(item_embeddings)
        # 1PL Model: logit = ability - difficulty
        logits = model_ability - item_difficulty
        return torch.sigmoid(logits)

    def get_all_probs(self, all_item_embeddings):
        # Helper to reconstruct the full response matrix for evaluation
        all_difficulties = self.difficulty_net(all_item_embeddings).squeeze()
        logits = self.thetas - all_difficulties[None, :]
        return torch.sigmoid(logits)

# ===================================================================
# == Step 3: Train the Model
# ===================================================================
N_MODELS = resmat.shape[0]
model = NeuralIRT1PL(N_MODELS, embed_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
loss_fn = nn.BCELoss() # Binary Cross-Entropy is best for 0/1 data

n_epochs = 50 # Adjust as needed
print(f"\n--- Step 3: Training Neural IRT model for {n_epochs} epochs ---")
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}")
    for model_idx, item_idx, responses in pbar:
        model_idx, item_idx, responses = model_idx.to(device), item_idx.to(device), responses.to(device)
        
        item_embeds = embeddings_tensor[item_idx]
        
        optimizer.zero_grad()
        predicted_probs = model(model_idx, item_embeds).squeeze()
        loss = loss_fn(predicted_probs, responses)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{n_epochs} | Average Training Loss: {avg_loss:.6f}")

# ===================================================================
# == Step 4: Evaluate the Final Model
# ===================================================================
print("\n--- Step 4: Evaluating the final reconstructed matrix ---")
model.eval()
with torch.no_grad():
    final_probs_tensor = model.get_all_probs(embeddings_tensor)

# Use the train/test masks created in Step 1 for the final evaluation
train_idtor_tensor = train_idtor.to(device)
test_idtor_tensor = test_idtor.to(device)

print("\n--- Final Evaluation of Neural IRT (1PL) ---")
compute_auc(final_probs_tensor, data_tensor, train_idtor_tensor, test_idtor_tensor)
compute_cttcorr(final_probs_tensor, data_tensor, train_idtor_tensor, test_idtor_tensor)